## 1. Setup and Installation

First, install the required dependencies and import the modules.

> **WARNING**: This notebook uses the unofficial `robin_stocks` library to interact with Robinhood's private API.
> Automated logins may violate Robinhood's Terms of Service. Use at your own risk.


In [ ]:
# If running for the first time, uncomment and run:
# !pip install -q robin-stocks pandas jupyter

# Add the parent directory to Python path to import our module
import sys
sys.path.insert(0, '..')

from src import (
    RobinhoodFetcher,
    RobinhoodFetcherBuilder,
    Stock,
    print_quote_summary,
    print_historical_summary,
    export_quote_to_csv,
    export_historical_to_csv,
    format_price,
    format_volume,
    format_percent
)

import pandas as pd
from datetime import datetime
import getpass

print('All imports successful!')


## 2. Choose Mode: Demo or Real Account

**Demo Mode** — No account needed. The notebook generates synthetic but realistic stock data.

**Real Account** — Uses your actual Robinhood credentials. Your credentials are **not stored** — they are only used during this session.

> **Security Note**: If using a real account, use a separate Robinhood account (not your main trading account) if possible.


In [ ]:
# Choose demo mode or real account
mode = input("Enter mode ('demo' for no account, or press Enter for real account): ").strip().lower()
USE_DEMO_MODE = (mode == 'demo' or mode == 'd')

if USE_DEMO_MODE:
    print("✓ Demo mode selected. No credentials needed.")
else:
    # Enter your Robinhood credentials
    # Using getpass so the password is not visible while typing
    ROBINHOOD_USERNAME = input("Robinhood email/username: ")
    ROBINHOOD_PASSWORD = getpass.getpass("Robinhood password: ")

    # Optionally enter MFA code if you have 2FA enabled
    mfa = input("MFA code (press Enter if not required): ").strip()
    ROBINHOOD_MFA = mfa if mfa else None

print("Ready to initialize fetcher...")


In [ ]:
# Initialize and login
if USE_DEMO_MODE:
    fetcher = RobinhoodFetcher(demo_mode=True)
    fetcher.login(username="", password="")
    print("✓ Demo mode activated! Using synthetic data.")
else:
    fetcher = RobinhoodFetcher()
    try:
        if ROBINHOOD_MFA:
            fetcher.login(
                username=ROBINHOOD_USERNAME,
                password=ROBINHOOD_PASSWORD,
                mfa_code=ROBINHOOD_MFA
            )
        else:
            fetcher.login(
                username=ROBINHOOD_USERNAME,
                password=ROBINHOOD_PASSWORD
            )
        print("Successfully logged in to Robinhood!")
    except Exception as e:
        print(f"Login failed: {e}")
        print("Troubleshooting:")
        print("  - Check your username and password")
        print("  - If you have 2FA enabled, provide an MFA code above")
        print("  - Robinhood may be rate-limiting repeated login attempts")
        print("  - Set mode to 'demo' above to run without a real account")

print(f"Logged in: {fetcher.logged_in}")


## 3. Account Overview

Let's check your Robinhood account information and current positions.

> **Note**: These are Robinhood-specific features not available via yfinance.


In [ ]:
# Fetch account information
account = fetcher.fetch_account_info()

if account:
    print("=" * 50)
    print("ACCOUNT SUMMARY")
    print("=" * 50)
    print(f"Account Number: {account.get('account_number', 'N/A')}")
    print(f"Buying Power:   {account.get('buying_power', 0):.2f}")
    print(f"Cash:           {account.get('cash', 0):.2f}")
    print(f"Portfolio Value: {account.get('portfolio_value', 0):.2f}")
    print(f"Equity:         {account.get('equity', 0):.2f}")
    print("=" * 50)
else:
    print("Could not fetch account info.")


In [ ]:
# Fetch current stock positions
positions = fetcher.fetch_positions()

if positions:
    print("=" * 65)
    print(f"CURRENT POSITIONS ({len(positions)} holdings)")
    print("=" * 65)

    for pos in positions:
        print(f"  {pos['symbol']:6s} | {pos['quantity']:>8.4f} shares | Avg: {pos['average_buy_price']:>8.2f} | P&L: {pos['equity_change']:>9.2f} ({pos['percent_change']:.2f}%)")
    print("=" * 65)
else:
    print("No positions found (empty portfolio or data unavailable).")


## 4. Fetching Stock Quotes

Now let's fetch current stock quotes. Robinhood provides **real-time** prices (not 15-min delayed like yfinance).


In [ ]:
# Fetch a single stock quote
print("Fetching AAPL stock quote from Robinhood...")
aapl = fetcher.fetch_quote("AAPL")

# Display the quote using the shared utility
print_quote_summary(aapl)


In [ ]:
# Access specific quote information
if aapl.quote:
    print(f"Stock: {aapl.symbol}")
    print(f"Current Price: {aapl.quote.price:.2f}")
    print(f"Price Change: {aapl.get_price_change():.2f} ({aapl.get_price_change_percent():.2f}%)")
    print(f"Today's High: {aapl.quote.high_price:.2f}")
    print(f"Today's Low: {aapl.quote.low_price:.2f}")
    print(f"Volume: {aapl.quote.volume:,}")


## 5. Analyzing Historical Data

Robinhood supports different historical data intervals. Note that the span/interval combinations differ from yfinance:

| Span | Available Intervals |
|------|-------------------|
| day | 5minute, 10minute |
| week | 10minute, hour |
| month | hour, day |
| 3month, year, 5year | day |


In [ ]:
# Fetch historical data (3 months, daily)
print("Fetching 3 months of daily historical data for AAPL...")
aapl_history = fetcher.fetch_historical_data("AAPL", period="3month", interval="day")

# Display summary using shared utility
print_historical_summary(aapl_history)


In [ ]:
# Convert historical data to pandas DataFrame for analysis
if aapl_history.historical_data:
    data = []
    for hd in aapl_history.historical_data:
        data.append({
            'Date': hd.date.date(),
            'Open': hd.open_price,
            'High': hd.high_price,
            'Low': hd.low_price,
            'Close': hd.close_price,
            'Volume': hd.volume,
            'Adj Close': hd.adjusted_close
        })

    df = pd.DataFrame(data)

    print("First 5 rows of historical data:")
    print(df.head())
    print(f"Total rows: {len(df)}")


In [ ]:
# Calculate some basic statistics
if aapl_history.historical_data:
    df_copy = df.copy()
    df_copy['Daily Change'] = df_copy['Close'].diff()
    df_copy['Daily Change %'] = df_copy['Close'].pct_change() * 100

    print("Price Statistics:")
    print(f"Min Close: {df['Close'].min():.2f}")
    print(f"Max Close: {df['Close'].max():.2f}")
    print(f"Mean Close: {df['Close'].mean():.2f}")
    print(f"Std Dev: {df['Close'].std():.2f}")
    print(f"Average Daily Volume: {int(df['Volume'].mean()):,}")
    print(f"Max Volume: {int(df['Volume'].max()):,}")


## 6. Intraday Data

Robinhood provides intraday data at finer granularity than yfinance. Let's fetch some hourly data for the past week.


In [ ]:
# Fetch intraday data (1 week, hourly)
print("Fetching 1 week of hourly data for AAPL...")
aapl_intraday = fetcher.fetch_historical_data("AAPL", period="week", interval="hour")

if aapl_intraday.historical_data:
    data_points = aapl_intraday.historical_data

    print(f"Intraday data points: {len(data_points)}")
    print("First 5 hourly entries:")
    for i in range(min(5, len(data_points))):
        h = data_points[i]
        print(f"  {h.date.strftime('%Y-%m-%d %H:%M')} | O: {h.open_price:>8.2f} | H: {h.high_price:>8.2f} | L: {h.low_price:>8.2f} | C: {h.close_price:>8.2f} | V: {h.volume:,}")

    # Calculate intraday volatility
    intraday_df = pd.DataFrame([{"DateTime": h.date, "Close": h.close_price} for h in data_points])
    intraday_vol = intraday_df["Close"].pct_change().std()
    print(f"Intraday Hourly Volatility: {intraday_vol:.6f}")
else:
    print("No intraday data available (markets may be closed).")


## 7. Tracking Multiple Stocks

Compare multiple stocks at once. Robinhood supports **batch quote fetching** in a single API call.


In [ ]:
# Fetch quotes for multiple stocks (single API call)
symbols = ["AAPL", "GOOGL", "MSFT", "AMZN", "TSLA", "NVDA", "META"]
print(f"Fetching quotes for {len(symbols)} symbols in one batch call...")

stocks = fetcher.fetch_multiple_quotes(symbols)

# Create a comparison table
stock_data = []
for symbol, stock in stocks.items():
    if stock.quote:
        stock_data.append({
            'Symbol': symbol,
            'Price': f'{stock.quote.price:.2f}',
            'Change': f'{stock.get_price_change():.2f}',
            'Change %': f'{stock.get_price_change_percent():.2f}%',
            'Open': f'{stock.quote.open_price:.2f}',
            'High': f'{stock.quote.high_price:.2f}',
            'Low': f'{stock.quote.low_price:.2f}',
            'Volume': f'{stock.quote.volume:,}'
        })

comparison_df = pd.DataFrame(stock_data)
print(comparison_df.to_string(index=False))


## 8. Exporting Data to CSV

The same shared utilities from `src.utils` work seamlessly with the Robinhood fetcher since both return `Stock` objects.


In [ ]:
# Fetch both quote and historical data
print("Fetching GOOGL quote and historical data...")
googl = fetcher.fetch_quote_and_history("GOOGL", period="3month", interval="day")

# Export quote to CSV
if googl and googl.quote:
    quote_file = export_quote_to_csv(googl)
    print(f"Quote exported to: {quote_file}")
else:
    print("Quote data not available.")

# Export historical data to CSV
if googl and googl.historical_data:
    history_file = export_historical_to_csv(googl)
    print(f"Historical data exported to: {history_file}")
else:
    print("No historical data available.")


In [ ]:
# Read back the exported CSV file
import os

if "history_file" in locals() and os.path.exists(history_file):
    df_exported = pd.read_csv(history_file)
    print("Exported data (first 5 rows):")
    print(df_exported.head())
    print(f"Total rows exported: {len(df_exported)}")


## 9. Advanced Analytics

Perform more advanced analysis - moving averages, volatility, and trend comparison.


In [ ]:
# Calculate moving averages
if aapl_history.historical_data:
    df_ma = df.copy()

    # 5-day and 20-day moving averages
    df_ma['MA_5'] = df_ma['Close'].rolling(window=5).mean()
    df_ma['MA_20'] = df_ma['Close'].rolling(window=20).mean()

    print("Moving Averages (Last 5 rows):")
    print(df_ma[['Date', 'Close', 'MA_5', 'MA_20']].tail())


In [ ]:
# Calculate annualized volatility
if aapl_history.historical_data:
    returns = df['Close'].pct_change()
    volatility = returns.std() * (252 ** 0.5)

    print("AAPL Volatility Analysis:")
    print(f"Daily Volatility: {returns.std():.4f}")
    print(f"Annualized Volatility: {volatility:.4f} or {volatility*100:.2f}%")
    print("Price Range (3 months):")
    print(f"Lowest: {df['Close'].min():.2f}")
    print(f"Highest: {df['Close'].max():.2f}")


## 10. Cleanup

Always log out of Robinhood when you are done to invalidate the session token.


In [ ]:
# Log out of Robinhood
fetcher.logout()
print("Logged out of Robinhood.")


## Summary

You have successfully used the Robinhood fetcher module to:

- Authenticate with Robinhood using your credentials
- View your account balance and buying power
- List current stock positions with P&L
- Fetch real-time stock quotes (not 15-min delayed)
- Retrieve historical and intraday price data
- Compare multiple stocks with batch quoting
- Export data to CSV
- Calculate moving averages and volatility
- Properly log out

### Key Differences from yfinance (StockFetcher):

| Feature | yfinance (StockFetcher) | Robinhood (RobinhoodFetcher) |
|---|---|---|
| Authentication | None required | Login required |
| Price delay | ~15 minutes | Real-time |
| Account data | No | Yes (balance, positions) |
| Intraday granularity | 1m, 5m, 15m, 30m, 60m | 5minute, 10minute, hour |
| Reliability | May break silently | Unofficial, may break |
| Risk | None | Account may be flagged |

### Next Steps:
1. Build an automated trading strategy using robin_stocks order endpoints
2. Set up real-time price streaming via websocket
3. Create visualization dashboards of your portfolio performance
